# mixup reproduction: pilot first

This notebook follows the submitted proposal. It does not contain experimental results.
Run it on a Colab GPU runtime, from VS Code connected to Colab (used for the 2026-10-28
pilot) or in the Colab browser interface. If the Drive cell fails from VS Code, use the browser.
Start with **Runtime > Change runtime type > GPU**. The pilot uses only development data.
Keep the full experiment and final-test switches off until you understand their consequences.

Put `mixup-colab-project.zip` in the top level of your Google Drive (`MyDrive`) so every new
session can reuse it. If it is not there, the next cell asks you to upload it instead.
It contains the project code, config, source attribution and tests. It contains no data,
model checkpoints, or personal PDFs.


In [1]:
#Environment check  

from pathlib import Path
import os, zipfile, subprocess, sys
import torch, torchvision, numpy
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130 torchvision: 0.26.0+cu130
GPU: Tesla T4


In [2]:
#Load project from Google Drive

from google.colab import drive
drive.mount("/content/drive")
ZIP = Path("/content/drive/MyDrive/mixup-colab-project.zip")
if not ZIP.exists():
    from google.colab import files
    uploaded = files.upload()
    assert "mixup-colab-project.zip" in uploaded, "Upload the project ZIP supplied with this notebook."
    ZIP = Path("mixup-colab-project.zip").resolve()
project = Path("/content/mixup-project")
project.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as archive:
    for item in archive.infolist():
        destination = (project / item.filename).resolve()
        assert destination.is_relative_to(project.resolve()), "Unsafe archive path"
    archive.extractall(project)
os.chdir(project)
print("Project:", project, "from", ZIP)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/mixup-project from /content/drive/MyDrive/mixup-colab-project.zip


## Save checkpoints outside the temporary runtime

The dataset lives in `/content` and can be downloaded again after a disconnect.
Runs and logs live on Drive. Use a new experiment directory for a revised protocol.
Do not run two copies of this notebook against the same output directory simultaneously.
Each checkpoint is about 135 MB. It is replaced every 5 epochs (`--save-every`) and at the
final epoch; a disconnect reruns the unsaved epochs identically. After the pilot, check that
Drive's trash is not filling up with replaced checkpoints (it counts toward your quota).


In [3]:
#Set up paths for running the project and testing it. 

RUN_ROOT = Path("/content/drive/MyDrive/mixup-reproduction-v1")
DATA = Path("/content/cifar10")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
def run(*args):
    # Stream output so test results and epoch lines appear in Colab while they run.
    with subprocess.Popen([sys.executable, "-u", *map(str, args)], stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="")
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, args)
run("-m", "unittest", "discover", "-s", "tests", "-v")

test_epoch_boundaries_and_tie_rule (test_protocol.ProtocolTests.test_epoch_boundaries_and_tie_rule) ... ok
test_evaluation_gate_and_validation_selected_weights (test_protocol.ProtocolTests.test_evaluation_gate_and_validation_selected_weights) ... erm seed=1 epoch=1/1 val_error=37.50% lr=0.1 compute=0.1s [checkpoint saved]
mixup seed=1 epoch=1/1 val_error=37.50% lr=0.1 compute=0.0s [checkpoint saved]
Frozen all runs. Evaluation may now load the test dataset.
Saved final test metrics. All values are from selected validation checkpoints.
ok
test_freeze_rejects_incomplete_runs_and_blocks_later_training (test_protocol.ProtocolTests.test_freeze_rejects_incomplete_runs_and_blocks_later_training) ... erm seed=1 epoch=1/2 val_error=37.50% lr=0.1 compute=0.0s [checkpoint saved]
mixup seed=1 epoch=1/2 val_error=37.50% lr=0.1 compute=0.0s [checkpoint saved]
erm seed=1 epoch=2/2 val_error=37.50% lr=0.01 compute=0.0s [checkpoint saved]
mixup seed=1 epoch=2/2 val_error=37.50% lr=0.01 compute=0.0s [ch

In [4]:
# Restore CIFAR-10 from the Drive cache (run before Pilot/Training)

import shutil
CACHE = Path("/content/drive/MyDrive/cifar10-cache/cifar-10-python.tar.gz")
if CACHE.exists() and not (DATA / "cifar-10-python.tar.gz").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    shutil.copy(CACHE, DATA)


## Read the protocol before running the pilot

The default is 50 epochs, batch 128, alpha 1, paired seeds 1/2/3, and a 45,000/5,000
stratified train/validation split. At this stage we only test feasibility. No test metrics are used.


In [4]:
import json
config = json.loads(Path("configs/proposal.json").read_text())
print(json.dumps(config, indent=2))
print(Path("docs/PROTOCOL.md").read_text())


{
  "experiment": "mixup-cifar10-proposal-v1",
  "epochs": 50,
  "batch_size": 128,
  "lr": 0.1,
  "momentum": 0.9,
  "weight_decay": 0.0001,
  "milestones": [
    25,
    38
  ],
  "alpha": 1.0,
  "seeds": [
    1,
    2,
    3
  ],
  "split_seed": 2026,
  "validation_per_class": 500,
  "num_workers": 2,
  "mean": [
    0.4914,
    0.4822,
    0.4465
  ],
  "std": [
    0.2023,
    0.1994,
    0.201
  ]
}
# Experiment protocol

Target: Section 3.2, Figure 3(a), PreAct ResNet-18 on CIFAR-10 in Zhang et al. (ICLR 2018).
Published errors 5.6% ERM / 4.2% mixup are reference values, never measured outputs here.

The submitted proposal is the authority for this implementation. The defaults are:

| Item | Decision |
|---|---|
| Dataset | CIFAR-10, 45,000 train / 5,000 validation / 10,000 test |
| Split | Stratified, 500 validation images per class, split seed 2026 |
| Model | Official repository ResNet18 constructor (PreActBlock), unchanged |
| Augmentation | Random crop 32 with padding 4, h

In [5]:
RUN_PILOT = True
if RUN_PILOT:
    run("-m", "mixup_repro.experiment", "pilot", "--download", "--resume",
        "--data", DATA, "--output", RUN_ROOT)


erm seed=1 epoch=1/2 val_error=87.50% lr=0.1 compute=2.1s
erm seed=1 epoch=2/2 val_error=82.42% lr=0.1 compute=1.4s [checkpoint saved]
mixup seed=1 epoch=1/2 val_error=88.67% lr=0.1 compute=1.5s
mixup seed=1 epoch=2/2 val_error=89.45% lr=0.1 compute=1.5s [checkpoint saved]
Pilot only: 1,024 train / 256 validation images. These are NOT reproduction results.


In [6]:
import csv
for method in ("erm", "mixup"):
    history = RUN_ROOT / "pilot" / f"{method}_seed1" / f"{method}_seed1_history.csv"
    if history.exists():
        print(method, list(csv.DictReader(history.open())))
print("Pilot timings exclude checkpoint writes. The full dataset is much larger.")
print("Bring these timing and GPU-memory outputs back before committing to the full budget.")


erm [{'epoch': '1', 'lr': '0.1', 'train_objective': '2.4833960831165314', 'validation_loss': '2.2909175157546997', 'validation_error_pct': '87.5', 'compute_seconds': '2.0940094890011096', 'peak_gpu_mb': '850.2685546875'}, {'epoch': '2', 'lr': '0.1', 'train_objective': '2.25424724817276', 'validation_loss': '3.552505612373352', 'validation_error_pct': '82.421875', 'compute_seconds': '1.4231928379995225', 'peak_gpu_mb': '850.2685546875'}]
mixup [{'epoch': '1', 'lr': '0.1', 'train_objective': '2.7192161083221436', 'validation_loss': '2.3524011373519897', 'validation_error_pct': '88.671875', 'compute_seconds': '1.5389657610012364', 'peak_gpu_mb': '853.01953125'}, {'epoch': '2', 'lr': '0.1', 'train_objective': '9.338361859321594', 'validation_loss': '9165937152.0', 'validation_error_pct': '89.453125', 'compute_seconds': '1.4528490100001363', 'peak_gpu_mb': '852.76953125'}]
Pilot timings exclude checkpoint writes. The full dataset is much larger.
Bring these timing and GPU-memory outputs bac

## Full reproduction (off by default)

Proceed after reviewing pilot feasibility. The full experiment needs six runs across one or more Colab sessions.
Set one METHOD/SEED pair at a time for easier supervision. Resume does not reset completed epochs.
If code/config changes, use a newly documented protocol and output directory.


In [5]:
#Training 
RUN_TRAINING = True
METHOD = "mixup"  # start with erm then "mixup"; repeat both for seeds 2 and 3
SEED = 3
MAIN = RUN_ROOT / "main"
if RUN_TRAINING:
    run("-m", "mixup_repro.experiment", "train", "--method", METHOD, "--seed", SEED,
        "--download", "--resume", "--save-every", 5, "--data", DATA, "--output", MAIN)


mixup seed=3 epoch=46/50 val_error=6.64% lr=0.001 compute=159.0s
mixup seed=3 epoch=47/50 val_error=6.34% lr=0.001 compute=88.8s
mixup seed=3 epoch=48/50 val_error=6.16% lr=0.001 compute=59.7s
mixup seed=3 epoch=49/50 val_error=6.64% lr=0.001 compute=60.0s
mixup seed=3 epoch=50/50 val_error=6.54% lr=0.001 compute=60.0s [checkpoint saved]


In [6]:
# Pre-freeze check: every run complete, checkpoint loads, CSV matches checkpoint (no test data)
import csv, torch
for method in ("erm", "mixup"):
    for seed in (1, 2, 3):
        folder = RUN_ROOT / "main" / f"{method}_seed{seed}"
        state = torch.load(folder / "checkpoint.pt", map_location="cpu", weights_only=False)
        with open(folder / f"{method}_seed{seed}_history.csv") as file:
            rows = list(csv.DictReader(file))
        epochs = [int(r["epoch"]) for r in rows]
        same = [float(r["validation_error_pct"]) for r in rows] == [h["validation_error_pct"] for h in state["history"]]
        slow = [e for e, r in zip(epochs, rows) if float(r["compute_seconds"]) > 80]
        print(f"{method} seed {seed}: checkpoint epoch {state['epoch']}, "
              f"best {state['best_error']:.2f}% at epoch {state['best_epoch']}, "
              f"epochs {epochs == list(range(1, 51))}, CSV matches checkpoint: {same}, slow epochs: {slow}")


erm seed 1: checkpoint epoch 50, best 7.20% at epoch 41, epochs True, CSV matches checkpoint: True, slow epochs: []
erm seed 2: checkpoint epoch 50, best 7.94% at epoch 40, epochs True, CSV matches checkpoint: True, slow epochs: []
erm seed 3: checkpoint epoch 50, best 6.04% at epoch 46, epochs True, CSV matches checkpoint: True, slow epochs: []
mixup seed 1: checkpoint epoch 50, best 17.98% at epoch 49, epochs True, CSV matches checkpoint: True, slow epochs: []
mixup seed 2: checkpoint epoch 50, best 8.82% at epoch 47, epochs True, CSV matches checkpoint: True, slow epochs: []
mixup seed 3: checkpoint epoch 50, best 6.16% at epoch 48, epochs True, CSV matches checkpoint: True, slow epochs: [21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]


In [7]:
# Mixup seed 3: runtimes recorded in the checkpoint, and epoch times 19-50 (no test data)
state = torch.load(RUN_ROOT / "main" / "mixup_seed3" / "checkpoint.pt", map_location="cpu", weights_only=False)
for env in state["environments"]:
    print(env["gpu"], "| python", env["python"], "| torch", env["torch"], "| cuda", env["cuda"])
print([(h["epoch"], round(h["compute_seconds"])) for h in state["history"][18:]])


Tesla T4 | python 3.13.16 | torch 2.11.0+cu130 | cuda 13.0
[(19, 60), (20, 60), (21, 127), (22, 126), (23, 126), (24, 126), (25, 125), (26, 126), (27, 126), (28, 126), (29, 126), (30, 124), (31, 126), (32, 126), (33, 126), (34, 126), (35, 124), (36, 126), (37, 126), (38, 126), (39, 126), (40, 126), (41, 125), (42, 126), (43, 126), (44, 125), (45, 125), (46, 159), (47, 89), (48, 60), (49, 60), (50, 60)]


In [6]:
#SAVE data to Google Drive
import shutil
CACHE = Path("/content/drive/MyDrive/cifar10-cache/cifar-10-python.tar.gz")
CACHE.parent.mkdir(exist_ok=True)
if not CACHE.exists():
    shutil.copy(DATA / "cifar-10-python.tar.gz", CACHE)
print("Cached:", CACHE.exists())


Cached: True


In [26]:
run("scripts/diagnose_mixup.py", "--output", RUN_ROOT / "main", "--data", DATA, "--download")


Seed 1, validation-selected weights. Errors in %. No test data was loaded.
| Method | Best epoch | Val (logged) | Val | Clean train | Val, BN re-est. | Clean train, BN re-est. |
|---|---:|---:|---:|---:|---:|---:|
| erm | 41 | 7.20 | 7.20 | 0.46 | 7.14 | 0.44 |
| mixup | 49 | 17.98 | 17.98 | 15.72 | 16.94 | 14.58 |
"Val" should equal "Val (logged)"; a difference means the evaluation is not reproducible.


In [30]:
run("-m", "mixup_repro.experiment", "train", "--config", "configs/check_small_alpha.json",
    "--method", "mixup", "--seed", 1, "--download", "--resume",
    "--data", DATA, "--output", RUN_ROOT / "checks" / "small_alpha")


mixup seed=1 epoch=1/5 val_error=64.02% lr=0.1 compute=51.0s
mixup seed=1 epoch=2/5 val_error=53.42% lr=0.1 compute=57.7s
mixup seed=1 epoch=3/5 val_error=43.80% lr=0.1 compute=59.9s
mixup seed=1 epoch=4/5 val_error=41.58% lr=0.1 compute=60.1s
mixup seed=1 epoch=5/5 val_error=31.16% lr=0.1 compute=59.3s [checkpoint saved]


## Freeze and final evaluation (off by default)

Only proceed after all six runs finish and every decision is fixed. Freeze verifies matching
configs, splits, source hashes, completion and paired initialization. It blocks further training.
The evaluator loads only validation-selected model weights and evaluates the untouched test split.


In [8]:
FREEZE_PROTOCOL = False
EVALUATE_TEST = False
if FREEZE_PROTOCOL:
    run("-m", "mixup_repro.experiment", "freeze", "--output", MAIN)
if EVALUATE_TEST:
    run("-m", "mixup_repro.experiment", "evaluate", "--download", "--data", DATA, "--output", MAIN)


In [9]:
if (MAIN / "test_results.json").exists():
    run("-m", "mixup_repro.summarize", "--output", MAIN)
    from IPython.display import Image, display, Markdown
    display(Markdown((MAIN / "summary.md").read_text()))
    display(Image(filename=str(MAIN / "learning_curves.png")))
else:
    print("No final test results yet. No scientific result is inferred from the pilot.")


No final test results yet. No scientific result is inferred from the pilot.
